# 04 — Condition effect analysis

This notebook reports **condition effect analyses** for the winning HDDM (`hddm_va_vat`):

1. **Group-level posterior condition effects** on drift rate (`v`) and decision boundary (`a`): main effects, interactions, and simple effects.
2. **Exploratory individual-level correlation** between Short−Long contrasts in `v` and `a` (posterior means).


- To recompute locally, place `hddm_va_vat_combined.nc` under `../models/transition_prob_duration_10000samples/` and set `RECOMPUTE_FROM_IDATA = True`, or run:
  ```bash
  python ../scripts/export_condition_effect_results.py --nc path/to/hddm_va_vat_combined.nc
  ```

## Prerequisites

- Prefer `hddm_analysis/notebooks/`; package root via `_resolve_hddm_root()`.
- Dependencies: `numpy`, `pandas`, `arviz`, `scipy` (see `../requirements.txt`). HDDM refitting is **not** required.
- Optional: run `02_model_comparison.ipynb` first with `LOAD_BEST_IDATA = True` to populate `best_idata` in memory.

## Methods note

- Condition effects use dummy-coded interaction decomposition; reference levels are the **last** level listed for each factor in `FACTORS`.
- Credible intervals are equal-tailed 95% posterior intervals.

In [1]:
import os
from pathlib import Path
import sys

def _resolve_hddm_root() -> Path:
    """Locate hddm_analysis/ regardless of notebook cwd."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "trials_hddm_ready.tsv").is_file():
            return candidate
        nested = candidate / "hddm_analysis"
        if (nested / "data" / "trials_hddm_ready.tsv").is_file():
            return nested
    if start.name == "notebooks" and (start.parent / "data").is_dir():
        return start.parent
    raise FileNotFoundError(
        "Could not locate hddm_analysis package root "
        "(expected data/trials_hddm_ready.tsv). "
        f"Started search from: {start}"
    )


ROOT = str(_resolve_hddm_root())
SCRIPTS_DIR = os.path.join(ROOT, "scripts")
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
DATA_DIR = os.path.join(ROOT, "data")
RESULTS_DIR = os.path.join(ROOT, "results")
MODEL_DIR = os.path.join(ROOT, "models", "transition_prob_duration_10000samples")
COND_EFFECT_DIR = os.path.join(RESULTS_DIR, "condition_effects")
FIG_DIR = os.path.join(ROOT, "figures")
for _d in (DATA_DIR, RESULTS_DIR, COND_EFFECT_DIR, MODEL_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)

WINNER = "hddm_va_vat"
RECOMPUTE_FROM_IDATA = False  # default: read precomputed CSVs
VERBOSE = False


In [2]:
from nc_io import from_netcdf as load_idata

idata = None
IDATA_PATH = os.path.join(MODEL_DIR, f"{WINNER}_combined.nc")

if "best_idata" in globals() and globals()["best_idata"] is not None:
    idata = globals()["best_idata"]
    print("Using best_idata from notebook 02.")
elif os.path.exists(IDATA_PATH):
    idata = load_idata(IDATA_PATH)
    print(f"Loaded InferenceData: {IDATA_PATH}")
elif RECOMPUTE_FROM_IDATA:
    raise FileNotFoundError(
        "RECOMPUTE_FROM_IDATA=True but no .nc found.\n"
        f"Expected: {IDATA_PATH}\n"
        "Or run notebook 02 with LOAD_BEST_IDATA=True."
    )
else:
    print(
        "No local InferenceData found — using precomputed condition-effect CSVs in results/condition_effects/."
    )


Loaded InferenceData: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\models\transition_prob_duration_10000samples\hddm_va_vat_combined.nc


## Part A — Group-level condition effects on drift rate (`v`) and decision boundary (`a`)

In [3]:
import pandas as pd

from posterior_contrasts import flexible_trace_analysis, results_to_rows

FACTORS = {
    "duration": ["Long", "Short"],
    "prob": ["prob_diff", "prob_same"],
    "transition": ["flexibility", "stability"],
}
PARAMS = ["v", "a"]


def contrasts_csv(param: str) -> str:
    return os.path.join(COND_EFFECT_DIR, f"{param}_posterior_contrasts.csv")


def print_contrast_table(df: pd.DataFrame, effect_prefix: str, title: str) -> None:
    subset = df[df["effect"].str.startswith(effect_prefix)]
    if subset.empty:
        return
    print(f"\n{title}")
    print("-" * 20)
    for _, row in subset.iterrows():
        print(f"{row['effect']}:")
        print(f"  Mean: {row['mean']:.4f}")
        print(f"  95% CI: [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")


def print_condition_effects(contrasts_df: pd.DataFrame, param: str) -> None:
    print(f"\nResults for parameter: {param} (model={WINNER})")
    print("=" * 50)
    print_contrast_table(contrasts_df, "main_effect", "Main Effects")
    print_contrast_table(contrasts_df, "interaction", "Interactions")
    print_contrast_table(contrasts_df, "simple_effect", "Simple Effects (Two-Way)")

    followups = contrasts_df[
        contrasts_df["effect"].str.startswith(("simple_interaction", "simple_main"))
    ]
    if not followups.empty:
        print("\nSimple Interactions / Simple Main Effects (Three-Way follow-ups)")
        print("-" * 20)
        for _, row in followups.iterrows():
            print(f"{row['effect']}:")
            print(f"  Mean: {row['mean']:.4f}")
            print(f"  95% CI: [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")


def load_or_compute_contrasts(param: str) -> pd.DataFrame:
    csv_path = contrasts_csv(param)
    if RECOMPUTE_FROM_IDATA and idata is not None:
        results = flexible_trace_analysis(
            idata, param=param, factors=FACTORS, verbose=VERBOSE
        )
        df = pd.DataFrame(results_to_rows(results, param=param, model=WINNER))
        df.to_csv(csv_path, index=False)
        print(f"Recomputed and saved: {csv_path}")
        return df
    if os.path.exists(csv_path):
        print(f"Loaded contrasts: {csv_path}")
        return pd.read_csv(csv_path)
    raise FileNotFoundError(
        f"Missing {csv_path}. Run export_condition_effect_results.py with a local `.nc` file."
    )


contrasts_by_param = {}
for param in PARAMS:
    contrasts_by_param[param] = load_or_compute_contrasts(param)
    display(contrasts_by_param[param])
    print_condition_effects(contrasts_by_param[param], param)


Loaded contrasts: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\results\condition_effects\v_posterior_contrasts.csv


,model,parameter,effect_type,effect,mean,ci_lower,ci_upper
0,hddm_va_vat,v,interaction,interaction_duration_prob_duration_Long_prob_p...,1.276103,0.415693,2.130376
1,hddm_va_vat,v,interaction,interaction_duration_prob_transition_duration_...,-0.502074,-1.356799,0.347718
2,hddm_va_vat,v,interaction,interaction_duration_transition_duration_Long_...,0.680102,-0.195709,1.547634
3,hddm_va_vat,v,interaction,interaction_prob_transition_prob_prob_diff_tra...,-0.689477,-1.569999,0.164058
4,hddm_va_vat,v,main_effect,main_effect_duration_Long_vs_Short,-0.248373,-0.469296,-0.030774
5,hddm_va_vat,v,main_effect,main_effect_prob_prob_diff_vs_prob_same,0.172258,-0.047024,0.383685
6,hddm_va_vat,v,main_effect,main_effect_transition_flexibility_vs_stability,2.067347,1.851882,2.291192
7,hddm_va_vat,v,simple_effect,simple_effect_duration_Long_vs_Short_at_prob_p...,0.070653,-0.205460,0.345132
8,hddm_va_vat,v,simple_effect,simple_effect_duration_Long_vs_Short_at_prob_p...,-0.567399,-0.900077,-0.235002
9,hddm_va_vat,v,simple_effect,simple_effect_duration_Long_vs_Short_at_transi...,-0.078347,-0.431822,0.276956



Results for parameter: v (model=hddm_va_vat)

Main Effects
--------------------
main_effect_duration_Long_vs_Short:
  Mean: -0.2484
  95% CI: [-0.4693, -0.0308]
main_effect_prob_prob_diff_vs_prob_same:
  Mean: 0.1723
  95% CI: [-0.0470, 0.3837]
main_effect_transition_flexibility_vs_stability:
  Mean: 2.0673
  95% CI: [1.8519, 2.2912]

Interactions
--------------------
interaction_duration_prob_duration_Long_prob_prob_diff_Short_prob_same:
  Mean: 1.2761
  95% CI: [0.4157, 2.1304]
interaction_duration_prob_transition_duration_Long_prob_prob_diff_transition_flexibility_Short_prob_same_stability:
  Mean: -0.5021
  95% CI: [-1.3568, 0.3477]
interaction_duration_transition_duration_Long_transition_flexibility_Short_stability:
  Mean: 0.6801
  95% CI: [-0.1957, 1.5476]
interaction_prob_transition_prob_prob_diff_transition_flexibility_prob_same_stability:
  Mean: -0.6895
  95% CI: [-1.5700, 0.1641]

Simple Effects (Two-Way)
--------------------
simple_effect_duration_Long_vs_Short_at_prob_pr

,model,parameter,effect_type,effect,mean,ci_lower,ci_upper
0,hddm_va_vat,a,interaction,interaction_duration_prob_duration_Long_prob_p...,-0.075899,-0.773750,0.627023
1,hddm_va_vat,a,interaction,interaction_duration_prob_transition_duration_...,0.137608,-0.549497,0.822391
2,hddm_va_vat,a,interaction,interaction_duration_transition_duration_Long_...,0.510974,-0.201864,1.226216
3,hddm_va_vat,a,interaction,interaction_prob_transition_prob_prob_diff_tra...,-0.260756,-0.982152,0.422905
4,hddm_va_vat,a,main_effect,main_effect_duration_Long_vs_Short,0.273253,0.093303,0.453440
5,hddm_va_vat,a,main_effect,main_effect_prob_prob_diff_vs_prob_same,-0.158149,-0.334093,0.010616
6,hddm_va_vat,a,main_effect,main_effect_transition_flexibility_vs_stability,2.674924,2.479062,2.884103
7,hddm_va_vat,a,simple_effect,simple_effect_duration_Long_vs_Short_at_prob_p...,0.254279,0.050875,0.461032
8,hddm_va_vat,a,simple_effect,simple_effect_duration_Long_vs_Short_at_prob_p...,0.292228,0.006365,0.586405
9,hddm_va_vat,a,simple_effect,simple_effect_duration_Long_vs_Short_at_transi...,0.400997,0.071554,0.731003



Results for parameter: a (model=hddm_va_vat)

Main Effects
--------------------
main_effect_duration_Long_vs_Short:
  Mean: 0.2733
  95% CI: [0.0933, 0.4534]
main_effect_prob_prob_diff_vs_prob_same:
  Mean: -0.1581
  95% CI: [-0.3341, 0.0106]
main_effect_transition_flexibility_vs_stability:
  Mean: 2.6749
  95% CI: [2.4791, 2.8841]

Interactions
--------------------
interaction_duration_prob_duration_Long_prob_prob_diff_Short_prob_same:
  Mean: -0.0759
  95% CI: [-0.7738, 0.6270]
interaction_duration_prob_transition_duration_Long_prob_prob_diff_transition_flexibility_Short_prob_same_stability:
  Mean: 0.1376
  95% CI: [-0.5495, 0.8224]
interaction_duration_transition_duration_Long_transition_flexibility_Short_stability:
  Mean: 0.5110
  95% CI: [-0.2019, 1.2262]
interaction_prob_transition_prob_prob_diff_transition_flexibility_prob_same_stability:
  Mean: -0.2608
  95% CI: [-0.9822, 0.4229]

Simple Effects (Two-Way)
--------------------
simple_effect_duration_Long_vs_Short_at_prob_pro

## Part B — Exploratory Δv vs Δa correlation (individual contrasts)

High-flux − low-flux mapping: **Short − Long**, averaged over the 2×2 (`prob` × `transition`) cells.


In [4]:
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import stats

CORR_CSV = os.path.join(COND_EFFECT_DIR, "delta_va_correlation_summary.csv")

_FLUX_FACTORS = {
    "prob": ["prob_diff", "prob_same"],
    "transition": ["flexibility", "stability"],
}
_SEP = "."


def _has_subject_nodes(inference_data) -> bool:
    return any("_subj" in str(k) for k in inference_data.posterior.data_vars)


def _cell_str(d, p, t):
    return _SEP.join([d, p, t])


def _subj_means_hddm_named(post, param, condition_str):
    pat = re.compile(
        "^" + re.escape(param) + r"_subj\(" + re.escape(condition_str) + r"\)\.(\d+)$"
    )
    out = {}
    for k in post.data_vars:
        m = pat.match(str(k))
        if not m:
            continue
        sid = int(m.group(1))
        x = np.asarray(post[k].values, dtype=float)
        out[sid] = float(np.mean(x, axis=tuple(range(x.ndim))))
    return out


def _subj_means_legacy(post, param, condition_str):
    stem = f"{param}({condition_str})"
    sk = f"{stem}_subj"
    if sk in post:
        x = np.asarray(post[sk].values, dtype=float)
        if x.ndim >= 3:
            m = np.mean(x, axis=tuple(range(x.ndim - 1)))
            return {i: float(m[i]) for i in range(m.shape[-1])}
        if x.ndim == 2:
            m = np.mean(x, axis=0)
            return {i: float(m[i]) for i in range(len(m))}
    out = {}
    for k in post.data_vars:
        m = re.fullmatch(re.escape(stem) + r"_subj\.(\d+)", str(k))
        if m:
            v = np.asarray(post[str(k)].values, dtype=float)
            out[int(m.group(1))] = float(np.mean(v, axis=tuple(range(v.ndim))))
    return out


def _subj_means(post, param, condition_str):
    for fn in (_subj_means_hddm_named, _subj_means_legacy):
        d = fn(post, param, condition_str)
        if d:
            return d
    raise KeyError(f"No subject traces for {param}_subj({condition_str}).<id>")


def _delta_short_minus_long(inference_data, param):
    post = inference_data.posterior
    sums = defaultdict(list)
    for p in _FLUX_FACTORS["prob"]:
        for t in _FLUX_FACTORS["transition"]:
            ds = _subj_means(post, param, _cell_str("Short", p, t))
            dl = _subj_means(post, param, _cell_str("Long", p, t))
            for sid in set(ds) & set(dl):
                sums[sid].append(ds[sid] - dl[sid])
    if not sums:
        raise ValueError("No overlapping Short/Long subject pairs in any cell.")
    return {sid: float(np.mean(vals)) for sid, vals in sums.items()}


def _save_correlation_summary(dv, da):
    r_pearson, p_pearson = stats.pearsonr(dv, da)
    summary = pd.DataFrame(
        [
            {
                "model": WINNER,
                "analysis": "delta_v_delta_a_short_minus_long",
                "n_subjects": len(dv),
                "pearson_r": float(r_pearson),
                "pearson_p": float(p_pearson),
                "note": (
                    "Exploratory Pearson correlation of posterior-mean individual contrasts "
                    "(Short − Long), averaged over 2×2 (prob × transition) cells."
                ),
            }
        ]
    )
    summary.to_csv(CORR_CSV, index=False)
    return summary


if RECOMPUTE_FROM_IDATA and idata is not None and _has_subject_nodes(idata):
    v_map = _delta_short_minus_long(idata, "v")
    a_map = _delta_short_minus_long(idata, "a")
    common = sorted(set(v_map) & set(a_map))
    dv = np.array([v_map[s] for s in common])
    da = np.array([a_map[s] for s in common])
    corr_df = _save_correlation_summary(dv, da)
    print("Recomputed individual-level correlation (local `.nc`; subject IDs not saved).")
elif os.path.exists(CORR_CSV):
    corr_df = pd.read_csv(CORR_CSV)
    print(f"Loaded correlation summary: {CORR_CSV}")
else:
    raise FileNotFoundError(
        f"Missing {CORR_CSV}. Regenerate with: python ../scripts/export_condition_effect_results.py"
    )

display(corr_df)
row = corr_df.iloc[0]
print(
    f"Pearson r = {row['pearson_r']:.4f} (p = {row['pearson_p']:.4g}), "
    f"n = {int(row['n_subjects'])}"
)


Loaded correlation summary: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\results\condition_effects\delta_va_correlation_summary.csv


,model,analysis,n_subjects,pearson_r,pearson_p,note
0,hddm_va_vat,delta_v_delta_a_short_minus_long,43,-0.254169,0.100028,Exploratory Pearson correlation of posterior-m...


Pearson r = -0.2542 (p = 0.1), n = 43
